<a href="https://colab.research.google.com/github/francisco-mreina/olist-Predict-Insatisfaction-and-delay/blob/main/03_modelizacion_predictiva.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis de clientes Olist (Brazilian E-Commerce)

##6. Modelización predictiva

###6.1.	Planteamiento del problema (Predicción de insatisfacción del cliente / Predicción de riesgo de retraso en la entrega)
###6.2.	Comparativa de modelos (Logistic Regression, Random Forest, XGBoost/LightGBM)
###6.3.	Comparativa de métricas (Accuracy, ROC-AUC, matriz de confusión) y métricas elegidas


###6.1.	Planteamiento del problema (Predicción de insatisfacción del cliente / Predicción de riesgo de retraso en la entrega)

In [ ]:
# fórmula de modelo completo OLS
def ols_formula(df, dependent_var, *excluded_cols):
    df_columns = list(df.columns.values)
    df_columns.remove(dependent_var)
    for col in excluded_cols:
        df_columns.remove(col)
    return dependent_var + ' ~ ' + ' + '.join(df_columns)

In [ ]:
# Función para calcular VIF
def vif_modelo(formula,data):

    # Matriz explícita de diseño y var objetivo por separado (extensión a dummies incluida)
    y, X = patsy.dmatrices(formula, data, return_type="dataframe")

    # Calcula VIF para las variables
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

    # Elimina la pendiente y ordena
    vif_tabla = vif_data[vif_data['feature'] != 'Intercept'].sort_values(by='VIF', ascending=False)

    return vif_tabla

####Insatisfacción
Primero hacemos Training-test split, usamos la fórmula OLS para la regresión lineal (primero de los modelos)

In [ ]:
predictores_insat_red = ['customer_region', 'n_sellers_distintos', 'pct_flete',
                         'ticket_medio_item', 'dias_aprobacion','order_purchase_hour'
]


In [ ]:
# 1. Training-test split
X_train_insat, X_test_insat, y_train_insat, y_test_insat = train_test_split(
    imput[predictores_insat_red], insatisfaccion, test_size=0.2, random_state=42
)

print('Training dataset shape Bin:', X_train_insat.shape, y_train_insat.shape)
print(y_train_insat.value_counts(normalize=True))

Training dataset shape Bin: (79552, 6) (79552,)
insatisfaccion
0    0.773205
1    0.226795
Name: proportion, dtype: float64


In [ ]:
# 2. Dataset de training con la objetivo dentro
data_train_insat = X_train_insat.copy().dropna()
data_train_insat['insatisfaccion'] = y_train_insat.loc[data_train_insat.index].astype('int64')

del X_train_insat, X_test_insat
gc.collect()

619

In [ ]:
# Aplicamos a fórmula de modelo reducido
form_insat = ols_formula(data_train_insat, 'insatisfaccion')
form_insat

'insatisfaccion ~ customer_region + n_sellers_distintos + pct_flete + ticket_medio_item + dias_aprobacion + order_purchase_hour'

In [ ]:
# Ajusto regresión de ejemplo
modelo_insat = logit(form_insat, data=data_train_insat).fit()
print(modelo_insat.summary())

Optimization terminated successfully.
         Current function value: 0.523554
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:         insatisfaccion   No. Observations:                78941
Model:                          Logit   Df Residuals:                    78931
Method:                           MLE   Df Model:                            9
Date:                Sat, 12 Sep 2026   Pseudo R-squ.:                 0.01110
Time:                        15:11:11   Log-Likelihood:                -41330.
converged:                       True   LL-Null:                       -41794.
Covariance Type:            nonrobust   LLR p-value:                7.000e-194
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept                      -3.0471      0.080    -38.162      0.000   

In [ ]:
form_insat = modelo_insat.model.formula
tabla_vif = vif_modelo(form_insat, data_train_insat)
tabla_vif

,feature,VIF
3,customer_region[T.Sudeste],4.046852
4,customer_region[T.Sur],2.961244
1,customer_region[T.Nordeste],2.390191
6,pct_flete,1.313563
2,customer_region[T.Norte],1.300402
7,ticket_medio_item,1.272719
8,dias_aprobacion,1.004012
5,n_sellers_distintos,1.000572
9,order_purchase_hour,1.000477


####Retraso
Primero hacemos Training-test split, usamos la fórmula OLS para la regresión lineal (primero de los modelos)

In [ ]:
predictores_retraso_red = [
    'customer_region',
    'n_items', 'n_pagos', 'max_installments', 'peso_total_g',
    'order_purchase_hour', 'ticket_medio_item', 'pct_flete'

]

In [ ]:
# 1. Training-test split — SOLO pedidos entregados
entregado_mask = imput['order_status'] == 'delivered'

X_train_retraso, X_test_retraso, y_train_retraso, y_test_retraso = train_test_split(
    imput.loc[entregado_mask, predictores_retraso_red],
    retraso.loc[entregado_mask],
    test_size=0.2, random_state=42
)

print('Training dataset shape Bin:', X_train_retraso.shape, y_train_retraso.shape)
print(y_train_retraso.value_counts(normalize=True))

Training dataset shape Bin: (77182, 8) (77182,)
retraso
0.0    0.918129
1.0    0.081871
Name: proportion, dtype: float64


In [ ]:
# 2. Dataset de training con la objetivo dentro
data_train_retraso = X_train_retraso.copy().dropna()
data_train_retraso['retraso'] = y_train_retraso.loc[data_train_retraso.index].astype('int64')

del X_train_retraso, X_test_retraso
gc.collect()

0

In [ ]:
# Aplicamos a fórmula de modelo reducido
form_retraso = ols_formula(data_train_retraso, 'retraso')
form_retraso

'retraso ~ customer_region + n_items + n_pagos + max_installments + peso_total_g + order_purchase_hour + ticket_medio_item + pct_flete'

In [ ]:
# Ajusto regresión de ejemplo
modelo_retraso = logit(form_retraso, data=data_train_retraso).fit()
print(modelo_retraso.summary())

Optimization terminated successfully.
         Current function value: 0.280675
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                retraso   No. Observations:                77181
Model:                          Logit   Df Residuals:                    77169
Method:                           MLE   Df Model:                           11
Date:                Sat, 12 Sep 2026   Pseudo R-squ.:                0.009224
Time:                        15:11:13   Log-Likelihood:                -21663.
converged:                       True   LL-Null:                       -21864.
Covariance Type:            nonrobust   LLR p-value:                 1.184e-79
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept                      -2.3229      0.092    -25.188      0.000   

In [ ]:
form_retraso = modelo_retraso.model.formula
tabla_vif = vif_modelo(form_retraso, data_train_retraso)
tabla_vif

,feature,VIF
3,customer_region[T.Sudeste],4.033556
4,customer_region[T.Sur],2.961338
1,customer_region[T.Nordeste],2.378965
10,ticket_medio_item,1.424071
11,pct_flete,1.348749
2,customer_region[T.Norte],1.303832
8,peso_total_g,1.167177
7,max_installments,1.166579
5,n_items,1.081354
6,n_pagos,1.002698


####Comparativa de modelos

In [ ]:
imput.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 38 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  category      
 1   customer_id                    99441 non-null  category      
 2   order_status                   99441 non-null  category      
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99295 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97660 non-null  datetime64[ns]
 6   order_delivered_customer_date  96484 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
 8   customer_unique_id             99441 non-null  category      
 9   customer_zip_code_prefix       99441 non-null  category      
 10  customer_city                  99441 non-null  category      
 11  customer_state 

In [ ]:
# RANDOM FOREST - INSATISFACCIÓN
# Variables predictoras
X = imput[predictores_insat_red].copy()
y = insatisfaccion.copy()

# Convertimos variables categóricas a dummies
X = pd.get_dummies(X, columns=['customer_region'], drop_first=True)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Random Forest modificado poco a poco
rf_insat = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=5,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
    )

# Entrenamiento
rf_insat.fit(X_train, y_train)

# Predicción
y_pred = rf_insat.predict(X_test)
y_prob = rf_insat.predict_proba(X_test)[:, 1]

# Evaluación
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.6987782191160943
ROC-AUC: 0.558869028373287

Matriz de confusión:
[[12786  2577]
 [ 3414  1112]]

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.83      0.81     15363
           1       0.30      0.25      0.27      4526

    accuracy                           0.70     19889
   macro avg       0.55      0.54      0.54     19889
weighted avg       0.68      0.70      0.69     19889



In [ ]:
# RANDOM FOREST - RETRASO
# Solo pedidos entregados
entregado_mask = imput['order_status'] == 'delivered'

# Variables predictoras y target
X = imput.loc[entregado_mask, predictores_retraso_red].copy()
y = retraso.loc[entregado_mask].copy()

# Convertimos la variable categórica a dummies
X = pd.get_dummies(X, columns=['customer_region'], drop_first=True)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Training dataset shape:', X_train.shape, y_train.shape)
print(y_train.value_counts(normalize=True))

# Random Forest modificado poco a poco
rf_retraso = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=5,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
    )

# Entrenamiento
rf_retraso.fit(X_train, y_train)

# Predicción
y_pred = rf_retraso.predict(X_test)
y_prob = rf_retraso.predict_proba(X_test)[:, 1]

# Evaluación
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Training dataset shape: (77182, 11) (77182,)
retraso
0.0    0.91888
1.0    0.08112
Name: proportion, dtype: float64
Accuracy: 0.8499170812603648
ROC-AUC: 0.5627177216921033

Matriz de confusión:
[[16146  1585]
 [ 1311   254]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.92      0.91      0.92     17731
         1.0       0.14      0.16      0.15      1565

    accuracy                           0.85     19296
   macro avg       0.53      0.54      0.53     19296
weighted avg       0.86      0.85      0.86     19296



### XGBOOST - INSATISFACCIÓN


In [ ]:
# Variables predictoras y target
X = imput[predictores_insat_red].copy()
y = insatisfaccion.copy()

# Convertimos variables categóricas a dummies
X = pd.get_dummies(X, columns=['customer_region'], drop_first=True)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Training dataset shape:', X_train.shape, y_train.shape)
print(y_train.value_counts(normalize=True))

peso_insat = (y_train == 0).sum() / (y_train == 1).sum()

# XGBoost modificado poco a poco
xgb_insat = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='aucpr',
 scale_pos_weight=peso_insat
    )

# Entrenamiento
xgb_insat.fit(X_train, y_train)

# Predicción
y_pred = xgb_insat.predict(X_test)
y_prob = xgb_insat.predict_proba(X_test)[:, 1]

# Evaluación
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Training dataset shape: (79552, 9) (79552,)
insatisfaccion
0    0.772463
1    0.227537
Name: proportion, dtype: float64
Accuracy: 0.616421137312082
ROC-AUC: 0.5607671863369271

Matriz de confusión:
[[10405  4958]
 [ 2671  1855]]

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.68      0.73     15363
           1       0.27      0.41      0.33      4526

    accuracy                           0.62     19889
   macro avg       0.53      0.54      0.53     19889
weighted avg       0.68      0.62      0.64     19889



In [ ]:
#Como los resultados son malos, intentamos mejorar con SKFold y cross_val_score de Nuestras Funciones

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(xgb_insat, X, y, cv=cv, scoring='roc_auc')
print(f"ROC-AUC (CV): {scores.mean():.3f} ± {scores.std():.3f}")

ROC-AUC (CV): 0.578 ± 0.005


###XGBoost — Retraso

In [ ]:
# Solo pedidos entregados
entregado_mask = imput['order_status'] == 'delivered'

# Variables predictoras y target
X = imput.loc[entregado_mask, predictores_retraso_red].copy()
y = retraso.loc[entregado_mask].copy()

# Convertimos variables categóricas a dummies
X = pd.get_dummies(X, columns=['customer_region'], drop_first=True)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Training dataset shape:', X_train.shape, y_train.shape)
print(y_train.value_counts(normalize=True))

peso_retraso = (y_train == 0).sum() / (y_train == 1).sum()

# XGBoost modificado poco a poco
xgb_retraso = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='aucpr',
 scale_pos_weight=peso_retraso
)

# Entrenamiento
xgb_retraso.fit(X_train, y_train)

# Predicción
y_pred = xgb_retraso.predict(X_test)
y_prob = xgb_retraso.predict_proba(X_test)[:, 1]

# Evaluación
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Training dataset shape: (77182, 11) (77182,)
retraso
0.0    0.91888
1.0    0.08112
Name: proportion, dtype: float64
Accuracy: 0.6921123548922057
ROC-AUC: 0.5730196909692109

Matriz de confusión:
[[12726  5005]
 [  936   629]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.93      0.72      0.81     17731
         1.0       0.11      0.40      0.17      1565

    accuracy                           0.69     19296
   macro avg       0.52      0.56      0.49     19296
weighted avg       0.86      0.69      0.76     19296



In [ ]:
#Como los resultados son malos, intentamos mejorar con SKFold y cross_val_score de Nuestras Funciones

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(xgb_retraso, X, y, cv=cv, scoring='roc_auc')
print(f"ROC-AUC (CV): {scores.mean():.3f} ± {scores.std():.3f}")

ROC-AUC (CV): 0.576 ± 0.006
